# CGC PSU Notebook

Documentation and manual test notebook for the CGC `PSU` wrapper.

Structure:
- one simple end-to-end sequence
- one PSU-only test-procedure section without AMX
- one connected-session section for wrapper methods that do not require `load_config()`
- one configured-session section for channel-control helpers
- one low-level transport section for `open_port()` / `close_port()` style calls


In [ ]:
from cgc.psu import PSU

DEVICE_ID = "PSU_TEST"
COM_PORT = 6
PORT_NUMBER = 0
BAUDRATE = 230400
STANDBY_CONFIG_NUMBER = 1
OPERATING_CONFIG_NUMBER = 2  # Example only: replace with your saved operating config.
TIMEOUT_S = 5.0

CHANNEL = 0
TEST_VOLTAGE_V = 5.0
TEST_CURRENT_A = 0.1

NO_LOAD_SET_VOLTAGE_V = 10.0
NO_LOAD_SET_CURRENT_A = 0.001
RUN_DUMMY_LOAD_OUTPUT_TEST = False
DUMMY_LOAD_TEST_VOLTAGE_V = 10.0
DUMMY_LOAD_TEST_CURRENT_A = 0.001

## 1. Simple Sequence

Recommended high-level sequence: instantiate, call `initialize()` with the standby
and operating configs, set one channel voltage/current pair, read back the outputs,
then shutdown. If the saved standby slot briefly leaves an HV output enabled,
`initialize()` now forces both outputs back OFF and verifies the readback
before it continues.


In [ ]:
psu_simple = PSU(device_id=DEVICE_ID, com=COM_PORT)  # Optional arguments: port=PORT_NUMBER, baudrate=BAUDRATE
startup_state = psu_simple.initialize(
    standby_config=STANDBY_CONFIG_NUMBER,
    operating_config=OPERATING_CONFIG_NUMBER,
    timeout_s=TIMEOUT_S,
)
print("startup_state", startup_state)
# Optional deeper startup check for maintenance or troubleshooting:
# print("psu_simple.collect_housekeeping()", psu_simple.collect_housekeeping())

In [ ]:
psu_simple.set_channel_voltage(CHANNEL, TEST_VOLTAGE_V)
psu_simple.set_channel_current(CHANNEL, TEST_CURRENT_A)
print("psu_simple.get_channel_voltage(CHANNEL)", psu_simple.get_channel_voltage(CHANNEL))
print("psu_simple.get_channel_current(CHANNEL)", psu_simple.get_channel_current(CHANNEL))

In [ ]:
psu_simple.shutdown()  # complete shutdown sequence, then disconnect.
psu_simple.close()  # releases Python-side resources

## 2. PSU-Only Test Procedures

These procedures are intended for a PSU that is disconnected from the AMX and from the experimental load.
Start with the HV outputs physically disconnected from the downstream setup.


### 2.1 Routine no-load health check

This checks communication, standby behavior, interlock state, and one housekeeping snapshot without intentionally driving HV into a load.


In [ ]:
psu_no_load = PSU(
    device_id=f"{DEVICE_ID}_no_load",
    com=COM_PORT,
    port=PORT_NUMBER,
    baudrate=BAUDRATE,
)
try:
    standby_state = psu_no_load.initialize(
        standby_config=STANDBY_CONFIG_NUMBER,
        timeout_s=TIMEOUT_S,
    )
    print("standby_state", standby_state)
    print("psu_no_load.get_interlock_enabled()", psu_no_load.get_interlock_enabled())
    snapshot = psu_no_load.collect_housekeeping()
    print("main_state", snapshot["main_state"])
    print("device_state", snapshot["device_state"])
    print("housekeeping", snapshot["housekeeping"])
finally:
    try:
        psu_no_load.shutdown()
    finally:
        psu_no_load.close()

### 2.2 Setpoint programming without AMX or load

You can program current and voltage setpoints without the AMX connected.
Keep the outputs disabled and read back the requested setpoints after writing them.


In [ ]:
psu_setpoints = PSU(
    device_id=f"{DEVICE_ID}_setpoints",
    com=COM_PORT,
    port=PORT_NUMBER,
    baudrate=BAUDRATE,
)
try:
    standby_state = psu_setpoints.initialize(
        standby_config=STANDBY_CONFIG_NUMBER,
        timeout_s=TIMEOUT_S,
    )
    print("standby_state", standby_state)
    psu_setpoints.set_channel_current(CHANNEL, NO_LOAD_SET_CURRENT_A)
    psu_setpoints.set_channel_voltage(CHANNEL, NO_LOAD_SET_VOLTAGE_V)
    print("current setpoint/limit", psu_setpoints.get_channel_current_limits(CHANNEL))
    print("voltage setpoint/limit", psu_setpoints.get_channel_voltage_limits(CHANNEL))
    print("outputs after programming", psu_setpoints.get_output_enabled())
finally:
    try:
        psu_setpoints.shutdown()
    finally:
        psu_setpoints.close()

### 2.3 Optional HV output test with dummy load only

Do not run this test into open circuit and do not run it through the AMX/electrode chain.
Enable it only when a suitable HV dummy load is connected and the safety setup has been reviewed.
This sequence keeps the output-control steps explicit instead of relying on an arbitrary operating config.


In [ ]:
if not RUN_DUMMY_LOAD_OUTPUT_TEST:
    print("Dummy-load HV output test is disabled. Set RUN_DUMMY_LOAD_OUTPUT_TEST = True only with a suitable HV dummy load.")
else:
    psu_dummy = PSU(
        device_id=f"{DEVICE_ID}_dummy",
        com=COM_PORT,
        port=PORT_NUMBER,
        baudrate=BAUDRATE,
    )
    try:
        standby_state = psu_dummy.initialize(
            standby_config=STANDBY_CONFIG_NUMBER,
            timeout_s=TIMEOUT_S,
        )
        print("standby_state", standby_state)
        psu_dummy.set_device_enabled(True)
        psu_dummy.set_output_enabled(False, False)
        psu_dummy.set_channel_current(0, 0.0)
        psu_dummy.set_channel_current(1, 0.0)
        psu_dummy.set_channel_voltage(0, 0.0)
        psu_dummy.set_channel_voltage(1, 0.0)
        psu_dummy.set_channel_current(CHANNEL, DUMMY_LOAD_TEST_CURRENT_A)
        if CHANNEL == 0:
            psu_dummy.set_output_enabled(True, False)
        else:
            psu_dummy.set_output_enabled(False, True)
        psu_dummy.set_channel_voltage(CHANNEL, DUMMY_LOAD_TEST_VOLTAGE_V)
        print("measured voltage", psu_dummy.get_channel_voltage(CHANNEL))
        print("measured current", psu_dummy.get_channel_current(CHANNEL))
        print("device_state", psu_dummy.collect_housekeeping()["device_state"])
    finally:
        try:
            psu_dummy.shutdown()
        finally:
            psu_dummy.close()

## 3. Connected Session

Use a dedicated instance to exercise connected-state wrapper methods one by one.
`connect()` is the intended entry point here.


In [ ]:
psu = PSU(
    device_id=f"{DEVICE_ID}_connected",
    com=COM_PORT,
    port=PORT_NUMBER,
    baudrate=BAUDRATE,
)

In [ ]:
print("psu.get_status()", psu.get_status())

In [ ]:
print("psu.connect(timeout_s=TIMEOUT_S)", psu.connect(timeout_s=TIMEOUT_S))

In [ ]:
print("psu.get_status()", psu.get_status())

In [ ]:
print("psu.format_status(psu.NO_ERR)", psu.format_status(psu.NO_ERR))

In [ ]:
print("psu.describe_error(psu.NO_ERR)", psu.describe_error(psu.NO_ERR))

In [ ]:
print("psu.get_product_info()", psu.get_product_info())

In [ ]:
print("psu.list_configs()", psu.list_configs())

In [ ]:
print("psu.get_main_state()", psu.get_main_state())

In [ ]:
print("psu.get_device_state()", psu.get_device_state())

In [ ]:
print("psu.get_housekeeping()", psu.get_housekeeping())

In [ ]:
print("psu.get_sensor_data()", psu.get_sensor_data())

In [ ]:
print("psu.get_fan_data()", psu.get_fan_data())

In [ ]:
print("psu.get_led_data()", psu.get_led_data())

In [ ]:
print("psu.get_cpu_data()", psu.get_cpu_data())

In [ ]:
print("psu.get_uptime()", psu.get_uptime())

In [ ]:
print("psu.get_total_time()", psu.get_total_time())

In [ ]:
print("psu.get_fw_version()", psu.get_fw_version())

In [ ]:
print("psu.get_fw_date()", psu.get_fw_date())

In [ ]:
print("psu.get_hw_type()", psu.get_hw_type())

In [ ]:
print("psu.get_hw_version()", psu.get_hw_version())

In [ ]:
print("psu.get_product_id()", psu.get_product_id())

In [ ]:
print("psu.get_product_no()", psu.get_product_no())

In [ ]:
print("psu.get_config_list()", psu.get_config_list())

In [ ]:
print("psu.get_config_name(OPERATING_CONFIG_NUMBER)", psu.get_config_name(OPERATING_CONFIG_NUMBER))

In [ ]:
print("psu.get_config_flags(OPERATING_CONFIG_NUMBER)", psu.get_config_flags(OPERATING_CONFIG_NUMBER))

In [ ]:
print("psu.get_device_enable()", psu.get_device_enable())

In [ ]:
print("psu.get_device_enabled()", psu.get_device_enabled())

In [ ]:
print("psu.get_output_enabled()", psu.get_output_enabled())

In [ ]:
print("psu.get_psu_enable()", psu.get_psu_enable())

In [ ]:
print("psu.get_psu_state()", psu.get_psu_state())

In [ ]:
print("psu.has_psu_full_range()", psu.has_psu_full_range())

In [ ]:
print("psu.get_psu_full_range()", psu.get_psu_full_range())

In [ ]:
print("psu.collect_housekeeping()", psu.collect_housekeeping())

In [ ]:
print("psu.disconnect()", psu.disconnect())

In [ ]:
psu.close()

## 4. Configured Session

Use a fresh configured instance for channel-control helpers.
This section also stages the PSU through the standby configuration first.
Run cleanup cells at the end of the section before reconnecting elsewhere.


In [ ]:
psu_oper = PSU(
    device_id=f"{DEVICE_ID}_oper",
    com=COM_PORT,
    port=PORT_NUMBER,
    baudrate=BAUDRATE,
)
print(
    "psu_oper.initialize(standby_config=STANDBY_CONFIG_NUMBER, timeout_s=TIMEOUT_S)",
    psu_oper.initialize(standby_config=STANDBY_CONFIG_NUMBER, timeout_s=TIMEOUT_S),
)

In [ ]:
print("psu_oper.load_config(OPERATING_CONFIG_NUMBER)", psu_oper.load_config(OPERATING_CONFIG_NUMBER))

In [ ]:
print("psu_oper.set_device_enabled(True)", psu_oper.set_device_enabled(True))

In [ ]:
print("psu_oper.get_device_enabled()", psu_oper.get_device_enabled())

In [ ]:
print("psu_oper.get_output_enabled()", psu_oper.get_output_enabled())

In [ ]:
print("psu_oper.set_output_enabled(True, True)", psu_oper.set_output_enabled(True, True))

In [ ]:
print("psu_oper.get_channel_voltage(CHANNEL)", psu_oper.get_channel_voltage(CHANNEL))

In [ ]:
print("psu_oper.get_channel_voltage_limits(CHANNEL)", psu_oper.get_channel_voltage_limits(CHANNEL))

In [ ]:
print("psu_oper.set_channel_voltage(CHANNEL, TEST_VOLTAGE_V)", psu_oper.set_channel_voltage(CHANNEL, TEST_VOLTAGE_V))

In [ ]:
print("psu_oper.get_channel_current(CHANNEL)", psu_oper.get_channel_current(CHANNEL))

In [ ]:
print("psu_oper.get_channel_current_limits(CHANNEL)", psu_oper.get_channel_current_limits(CHANNEL))

In [ ]:
print("psu_oper.set_channel_current(CHANNEL, TEST_CURRENT_A)", psu_oper.set_channel_current(CHANNEL, TEST_CURRENT_A))

In [ ]:
print("psu_oper.get_adc_housekeeping(CHANNEL)", psu_oper.get_adc_housekeeping(CHANNEL))

In [ ]:
print("psu_oper.get_psu_housekeeping(CHANNEL)", psu_oper.get_psu_housekeeping(CHANNEL))

In [ ]:
print("psu_oper.get_psu_data(CHANNEL)", psu_oper.get_psu_data(CHANNEL))

In [ ]:
print("psu_oper.get_psu_output_voltage(CHANNEL)", psu_oper.get_psu_output_voltage(CHANNEL))

In [ ]:
print("psu_oper.get_psu_set_output_voltage(CHANNEL)", psu_oper.get_psu_set_output_voltage(CHANNEL))

In [ ]:
print("psu_oper.get_psu_output_current(CHANNEL)", psu_oper.get_psu_output_current(CHANNEL))

In [ ]:
print("psu_oper.get_psu_set_output_current(CHANNEL)", psu_oper.get_psu_set_output_current(CHANNEL))

### Optional / Disruptive PSU Calls

The cells below intentionally change controller state more aggressively.
Uncomment them only when that behavior is desired on the hardware under test.


In [ ]:
# print("psu_oper.set_psu_full_range(True, True)", psu_oper.set_psu_full_range(True, True))

In [ ]:
# print("psu_oper.save_current_config(OPERATING_CONFIG_NUMBER)", psu_oper.save_current_config(OPERATING_CONFIG_NUMBER))

In [ ]:
psu_oper.shutdown()

In [ ]:
psu_oper.close()

## 5. Low-Level Transport Session

Dedicated instance for the explicit transport primitives. Do not mix this
section with `connect()` on the same object. On this low-level object,
`close()` only releases Python resources. Before `close_port()`, explicitly
drive the PSU to a safe state by zeroing both channels and disabling the
outputs and device.


In [ ]:
psu_low = PSU(
    device_id=f"{DEVICE_ID}_low",
    com=COM_PORT,
    port=PORT_NUMBER,
    baudrate=BAUDRATE,
)

In [ ]:
print("psu_low.open_port(COM_PORT, PORT_NUMBER)", psu_low.open_port(COM_PORT, PORT_NUMBER))

In [ ]:
print("psu_low.set_baud_rate(BAUDRATE)", psu_low.set_baud_rate(BAUDRATE))

In [ ]:
print("psu_low.purge()", psu_low.purge())

In [ ]:
print("psu_low.device_purge()", psu_low.device_purge())

In [ ]:
print("psu_low.get_buffer_state()", psu_low.get_buffer_state())

In [ ]:
print("psu_low.load_current_config(OPERATING_CONFIG_NUMBER)", psu_low.load_current_config(OPERATING_CONFIG_NUMBER))

In [ ]:
print("psu_low.reset_current_config()", psu_low.reset_current_config())

In [ ]:
print("psu_low.get_device_enable()", psu_low.get_device_enable())

In [ ]:
device_enable_status, current_enable = psu_low.get_device_enable()

In [ ]:
print("psu_low.set_device_enable(current_enable)", psu_low.set_device_enable(current_enable))

In [ ]:
print("psu_low.get_psu_enable()", psu_low.get_psu_enable())

In [ ]:
psu_enable_status, current_psu0, current_psu1 = psu_low.get_psu_enable()

In [ ]:
print("psu_low.set_psu_enable(current_psu0, current_psu1)", psu_low.set_psu_enable(current_psu0, current_psu1))

In [ ]:
print("psu_low.get_psu_full_range()", psu_low.get_psu_full_range())

In [ ]:
full_range_status, current_range0, current_range1 = psu_low.get_psu_full_range()

In [ ]:
print("psu_low.set_psu_full_range(current_range0, current_range1)", psu_low.set_psu_full_range(current_range0, current_range1))

In [ ]:
voltage_status, current_voltage = psu_low.get_psu_output_voltage(CHANNEL)

In [ ]:
print("psu_low.set_psu_output_voltage(CHANNEL, current_voltage)", psu_low.set_psu_output_voltage(CHANNEL, current_voltage))

In [ ]:
current_status, current_current = psu_low.get_psu_output_current(CHANNEL)

In [ ]:
print("psu_low.set_psu_output_current(CHANNEL, current_current)", psu_low.set_psu_output_current(CHANNEL, current_current))

In [ ]:
for ch in range(2):
    print(
        f"psu_low.set_psu_output_current({ch}, 0.0)",
        psu_low.set_psu_output_current(ch, 0.0),
    )
for ch in range(2):
    print(
        f"psu_low.set_psu_output_voltage({ch}, 0.0)",
        psu_low.set_psu_output_voltage(ch, 0.0),
    )
print("psu_low.set_psu_enable(False, False)", psu_low.set_psu_enable(False, False))
print("psu_low.set_device_enable(False)", psu_low.set_device_enable(False))
print("psu_low.close_port()", psu_low.close_port())

In [ ]:
psu_low.close()